In [1]:
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import tool
from langgraph.graph import StateGraph
from typing import Annotated,TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage,BaseMessage
from langgraph.prebuilt import ToolNode,tools_condition 

c:\Users\nk110\Desktop\AI\Langgraph\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
load_dotenv()

llm = ChatOpenAI(model="gpt-3.5-turbo",temperature=0.9)

loader = PyPDFLoader("NikhilNewResume.pdf")

docs = loader.load()
len(docs)

1

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,chunk_overlap=200)

chunks = splitter.split_documents(docs)
len(chunks)

3

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small",chunk_size=1)
vectorstore = FAISS.from_documents(chunks,embeddings)

vectorstore

In [6]:
retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":4} )

In [7]:
@tool
def rag_tool(query):

    """A tool that uses Retrieval Augmented Generation (RAG) to answer questions based on a given query. It retrieves relevant documents from a vector store and provides context and metadata for the query."""

    result = retriever.invoke(query)

    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]
    return {query: query, "context": context, "metadata": metadata}